# Avance 02 — C5 · Etapa 2: validación de calidad
**Equipo A&C · Inteligencia de Negocios**

En esta etapa **revisamos sin eliminar ni modificar los datos de la Etapa 1**. Analizamos 1,341,419 registros conservados y 76,204 discrepancias de alcaldía reportadas en la ejecución previa. Estos son **resultados de referencia**, no cifras precargadas como conclusiones: las comprobamos al ejecutar.

**Unidad de observación:** registro de incidente vial C5; no lo equiparamos automáticamente con un hecho SSC ni con personas lesionadas o fallecidas.

**Nota técnica:** procesamos el CSV por bloques para reducir el consumo de memoria en Anaconda JupyterLab. Los conteos de duplicados se calculan sobre las **17 columnas originales** (sin incluir procedencia, normalizaciones ni indicadores añadidos). Los posibles duplicados son señales de revisión, no filas para eliminar automáticamente.

## 1. Localización de la salida de la Etapa 1
Buscamos `incidentes_viales_C5_etapa1.csv` dentro de `resultados_C5/etapa1`. Si existen varias copias, solicitamos seleccionar una ruta para evitar mezclar versiones.

In [1]:
from pathlib import Path
from datetime import date
import pandas as pd
import numpy as np
import re
from collections import Counter, defaultdict
from IPython.display import display

RUTA_ETAPA1_MANUAL = None  # Ejemplo: Path('/home/usuario/proyecto/resultados/resultados_C5/etapa1')
TAM_BLOQUE = 50000
NOMBRE = 'incidentes_viales_C5_etapa1.csv'

if RUTA_ETAPA1_MANUAL is not None:
    ENTRADA = Path(RUTA_ETAPA1_MANUAL).expanduser().resolve()
else:
    cwd = Path.cwd().resolve()
    candidatas = [p / 'resultados' / 'resultados_C5' / 'etapa1'
                  for p in [cwd, *cwd.parents]]
    candidatas += [p / 'resultados_C5' / 'etapa1'
                   for p in [cwd, *cwd.parents]]
    candidatas = [p for p in candidatas if (p / NOMBRE).is_file()]
    if not candidatas:
        candidatas = [p for p in Path.home().rglob('etapa1')
                      if p.parent.name == 'resultados_C5' and (p / NOMBRE).is_file()]
    candidatas = sorted(set(p.resolve() for p in candidatas))
    if not candidatas:
        raise FileNotFoundError('No se encontró el CSV de Etapa 1. Configura RUTA_ETAPA1_MANUAL.')
    if len(candidatas) != 1:
        print('Hay varias carpetas candidatas:')
        for p in candidatas: print('-', p)
        raise RuntimeError('Selecciona la carpeta correcta en RUTA_ETAPA1_MANUAL.')
    ENTRADA = candidatas[0]

ARCHIVO = ENTRADA / NOMBRE
SALIDA = ENTRADA.parent / 'etapa2'
SALIDA.mkdir(parents=True, exist_ok=True)
print('Archivo:', ARCHIVO)
print('Resultados:', SALIDA)
print('Tamaño aproximado (MB):', round(ARCHIVO.stat().st_size / 1e6, 1))

Archivo: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/resultados/resultados_C5/etapa1/incidentes_viales_C5_etapa1.csv
Resultados: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/resultados/resultados_C5/etapa2
Tamaño aproximado (MB): 411.3


## 2. Contrato de lectura y columnas
Conservamos folios y códigos como texto; evitamos interpretar folios como números. No reemplazamos datos en el archivo original. Identificamos la procedencia y las columnas de trabajo creadas en la Etapa 1.

In [2]:
ORIGINALES = [
    'folio','fecha_creacion','hora_creacion','dia_semana','fecha_cierre',
    'hora_cierre','tipo_incidente_c4','incidente_c4','alcaldia_inicio',
    'codigo_cierre','clas_con_f_alarma','tipo_entrada','alcaldia_cierre',
    'alcaldia_catalogo','colonia_catalogo','longitud','latitud'
]
COLUMNAS_ETAPA1 = [
    'archivo_origen','fila_origen','fecha_creacion_validada',
    'alcaldia_catalogo_norm','alcaldia_inicio_norm','alcaldia_cierre_norm',
    'alcaldia_normalizada','alcaldias_discrepantes','motivo_exclusion'
]
cabecera = pd.read_csv(ARCHIVO, nrows=0, encoding='utf-8-sig').columns.tolist()
faltantes = [c for c in ORIGINALES + COLUMNAS_ETAPA1 if c not in cabecera]
if faltantes:
    raise ValueError('Faltan columnas esperadas de Etapa 1: ' + ', '.join(faltantes))
print('Columnas detectadas:', len(cabecera))
display(pd.DataFrame({'columna': cabecera, 'tipo_lectura': 'texto (str)'}))

Columnas detectadas: 26


,columna,tipo_lectura
0,folio,texto (str)
1,fecha_creacion,texto (str)
2,hora_creacion,texto (str)
3,dia_semana,texto (str)
4,fecha_cierre,texto (str)
5,hora_cierre,texto (str)
6,tipo_incidente_c4,texto (str)
7,incidente_c4,texto (str)
8,alcaldia_inicio,texto (str)
9,codigo_cierre,texto (str)


## 3. Reglas de validación
- **Fechas:** comprobamos formato `%Y-%m-%d`, rango desde 2018 y fechas futuras como alerta, sin eliminación.
- **Horas:** revisamos formato `HH:MM:SS` o `HH:MM`, rango 00:00–23:59 y ausencia de hora.
- **Alcaldías:** verificamos la alcaldía normalizada y los desacuerdos entre inicio, cierre y catálogo. Conservamos el valor de catálogo utilizado en la Etapa 1, pero la discrepancia **no queda resuelta**.
- **Folios:** separamos vacíos, `SD`/equivalentes y folios repetidos; un folio no equivale a una fila única.
- **Duplicados:** buscamos coincidencias en las 17 columnas originales, sin borrar ninguna.
- **Coordenadas:** comprobamos números y rangos globales; las coordenadas plausibles fuera de CDMX requieren un análisis espacial posterior.
- **Campos:** medimos vacíos y valores distintos por columna.
- **Cobertura:** analizamos año, mes, archivo de procedencia y categorías `incidente_c4`.

## 4. Validación por bloques
Este paso puede tardar varios minutos porque procesa más de un millón de registros. Mostramos avances cada cinco bloques. **No exportamos el archivo entero otra vez**, sino resúmenes y muestras acotadas de hallazgos.

Para duplicados exactos usamos una huella estable de las 17 columnas originales. Si posteriormente se pretende eliminar registros, debemos contrastar los casos contra las filas completas y confirmar la unidad de observación; las huellas no son una autorización de borrado.

In [3]:
ALCALDIAS = {
    'ALVARO OBREGON','AZCAPOTZALCO','BENITO JUAREZ','COYOACAN',
    'CUAJIMALPA DE MORELOS','CUAUHTEMOC','GUSTAVO A MADERO',
    'IZTACALCO','IZTAPALAPA','LA MAGDALENA CONTRERAS','MIGUEL HIDALGO',
    'MILPA ALTA','TLAHUAC','TLALPAN','VENUSTIANO CARRANZA','XOCHIMILCO'
}
SIN_FOLIO = {'', 'SD', 'S/D', 'N/D', 'ND', 'SIN DATO', 'SIN DATOS', 'NA', 'N/A'}
def vacio(s):
    t = s.astype('string').str.strip()
    return t.isna() | t.eq('').fillna(False)
def horas_validas(s):
    t = s.astype('string').str.strip()
    partes = t.str.extract(r'^(\d{1,2}):(\d{2})(?::(\d{2}))?$')
    h = pd.to_numeric(partes[0], errors='coerce')
    m = pd.to_numeric(partes[1], errors='coerce')
    seg = pd.to_numeric(partes[2], errors='coerce').fillna(0)
    return h.between(0,23) & m.between(0,59) & seg.between(0,59)

metricas = Counter()
archivos = Counter()
anios = Counter()
meses = Counter()
incidentes = Counter()
folios = Counter()
alcaldias = Counter()
faltantes = Counter()
valores_distintos = {c: set() for c in cabecera if c not in ['folio','fila_origen','latitud','longitud']}
huellas_vistas = set()
huellas_repetidas = set()
muestras = defaultdict(list)
MAX_MUESTRAS = 100
hoy = pd.Timestamp(date.today())

for num_bloque, bloque in enumerate(pd.read_csv(
        ARCHIVO, dtype=str, encoding='utf-8-sig', low_memory=False,
        chunksize=TAM_BLOQUE), start=1):
    metricas['filas'] += len(bloque)
    for col in cabecera:
        faltantes[col] += int(vacio(bloque[col]).sum())
    for col, distintos in valores_distintos.items():
        distintos.update(bloque[col].dropna().astype(str).unique().tolist())
    archivos.update(bloque['archivo_origen'].fillna('<NULO>').value_counts().to_dict())
    incidentes.update(bloque['incidente_c4'].fillna('<NULO>').value_counts().to_dict())
    alcaldias.update(bloque['alcaldia_normalizada'].fillna('<NULO>').value_counts().to_dict())

    fecha = pd.to_datetime(bloque['fecha_creacion'], format='%Y-%m-%d', errors='coerce')
    invalida = fecha.isna()
    anterior = fecha.notna() & fecha.lt(pd.Timestamp('2018-01-01'))
    futura = fecha.notna() & fecha.gt(hoy)
    metricas['fechas_invalidas'] += int(invalida.sum())
    metricas['fechas_antes_2018'] += int(anterior.sum())
    metricas['fechas_futuras_alerta'] += int(futura.sum())
    anios.update(fecha.dt.year.dropna().astype(int).value_counts().to_dict())
    meses.update(fecha.dt.strftime('%Y-%m').dropna().value_counts().to_dict())

    inicio_vacio = vacio(bloque['hora_creacion'])
    cierre_vacio = vacio(bloque['hora_cierre'])
    hora_ini_mal = ~inicio_vacio & ~horas_validas(bloque['hora_creacion'])
    hora_cie_mal = ~cierre_vacio & ~horas_validas(bloque['hora_cierre'])
    for clave, mascara in [
        ('hora_creacion_faltante',inicio_vacio), ('hora_creacion_invalida',hora_ini_mal),
        ('hora_cierre_faltante',cierre_vacio), ('hora_cierre_invalida',hora_cie_mal)
    ]: metricas[clave] += int(mascara.sum())

    alc_invalida = ~bloque['alcaldia_normalizada'].isin(ALCALDIAS)
    cols_alc = ['alcaldia_catalogo_norm','alcaldia_inicio_norm','alcaldia_cierre_norm']
    desacuerdo = bloque[cols_alc].apply(
        lambda f: len(set(f.dropna())) > 1, axis=1)
    metricas['alcaldia_normalizada_invalida'] += int(alc_invalida.sum())
    metricas['alcaldias_discrepantes'] += int(desacuerdo.sum())
    for c in cols_alc:
        metricas[c + '_faltante'] += int(vacio(bloque[c]).sum())

    folio = bloque['folio'].astype('string').str.strip().str.upper()
    vacios_folio = vacio(bloque['folio'])
    folio_sd = folio.isin(SIN_FOLIO) & ~vacios_folio
    metricas['folios_vacios'] += int(vacios_folio.sum())
    metricas['folios_sd_o_equivalentes'] += int(folio_sd.sum())
    folios.update(folio[~vacios_folio & ~folio_sd].value_counts().to_dict())

    lon = pd.to_numeric(bloque['longitud'], errors='coerce')
    lat = pd.to_numeric(bloque['latitud'], errors='coerce')
    metricas['coordenadas_no_numericas_o_faltantes'] += int((lon.isna() | lat.isna()).sum())
    metricas['coordenadas_fuera_rango_global'] += int(
        ((lon.notna() & ~lon.between(-180,180)) |
         (lat.notna() & ~lat.between(-90,90))).sum())

    # El hash usa solo las 17 columnas originales y representa coincidencia exacta de valores.
    huellas = pd.util.hash_pandas_object(
        bloque[ORIGINALES].fillna('<VALOR_NULO>'), index=False)
    repetidas_bloque = []
    for h in huellas.to_numpy(dtype='uint64'):
        k = int(h)
        repetidas_bloque.append(k in huellas_vistas)
        if k in huellas_vistas:
            huellas_repetidas.add(k)
        else:
            huellas_vistas.add(k)
    repetidas_bloque = pd.Series(repetidas_bloque, index=bloque.index)
    metricas['duplicados_adicionales_17_columnas'] += int(repetidas_bloque.sum())

    alertas = {
        'alcaldias_discrepantes': desacuerdo,
        'duplicados_adicionales': repetidas_bloque,
        'fecha_invalida': invalida,
        'fecha_futura': futura,
        'hora_creacion_invalida': hora_ini_mal,
        'coordenadas_invalidas': lon.isna() | lat.isna() |
            ~lon.between(-180,180) | ~lat.between(-90,90)
    }
    for clave, mascara in alertas.items():
        restante = MAX_MUESTRAS - len(muestras[clave])
        if restante > 0:
            seleccion = bloque.loc[mascara].head(restante).copy()
            if not seleccion.empty:
                seleccion.insert(0, 'tipo_alerta', clave)
                muestras[clave].extend(seleccion.to_dict('records'))
    if num_bloque % 5 == 0:
        print('Bloques procesados:', num_bloque, '| filas:', f"{metricas['filas']:,}")

print('Validación terminada. Filas revisadas:', f"{metricas['filas']:,}")

Bloques procesados: 5 | filas: 250,000
Bloques procesados: 10 | filas: 500,000
Bloques procesados: 15 | filas: 750,000
Bloques procesados: 20 | filas: 1,000,000
Bloques procesados: 25 | filas: 1,250,000
Validación terminada. Filas revisadas: 1,341,419


## 5. Resultados e indicadores de calidad
Resumimos los indicadores para tomar decisiones documentadas. Un registro puede presentar **varias alertas**; por tanto, los conteos de alertas no se suman como si fueran filas diferentes.

In [4]:
resumen = pd.DataFrame([
    {'indicador': k, 'filas': int(v), 'porcentaje_sobre_total':
     round(100 * v / metricas['filas'], 2)}
    for k, v in sorted(metricas.items()) if k != 'filas'
])
display(resumen)
resumen.to_csv(SALIDA / 'resumen_validacion_C5.csv', index=False, encoding='utf-8-sig')

perfil = pd.DataFrame([
    {'columna': col, 'faltantes_o_vacios': faltantes[col],
     'porcentaje_faltantes': round(100*faltantes[col]/metricas['filas'],2),
     'valores_distintos_aproximados': len(valores_distintos[col])
       if col in valores_distintos else pd.NA}
    for col in cabecera
])
display(perfil)
perfil.to_csv(SALIDA / 'perfil_columnas_C5.csv', index=False, encoding='utf-8-sig')

,indicador,filas,porcentaje_sobre_total
0,alcaldia_catalogo_norm_faltante,2345,0.17
1,alcaldia_cierre_norm_faltante,84,0.01
2,alcaldia_inicio_norm_faltante,102,0.01
3,alcaldia_normalizada_invalida,0,0.00
4,alcaldias_discrepantes,76204,5.68
5,coordenadas_fuera_rango_global,0,0.00
6,coordenadas_no_numericas_o_faltantes,0,0.00
7,duplicados_adicionales_17_columnas,0,0.00
8,fechas_antes_2018,0,0.00
9,fechas_futuras_alerta,0,0.00


,columna,faltantes_o_vacios,porcentaje_faltantes,valores_distintos_aproximados
0,folio,0,0.00,<NA>
1,fecha_creacion,0,0.00,2251
2,hora_creacion,1544,0.12,86303
3,dia_semana,0,0.00,7
4,fecha_cierre,0,0.00,2251
5,hora_cierre,1544,0.12,86359
6,tipo_incidente_c4,0,0.00,7
7,incidente_c4,0,0.00,17
8,alcaldia_inicio,102,0.01,16
9,codigo_cierre,0,0.00,5


## 6. Folios repetidos y duplicados exactos
**Folio repetido no significa duplicado exacto.** Documentamos cuántos folios válidos se repiten y cuántas filas se asocian con ellos. Las coincidencias exactas se miden sobre los 17 campos originales; si existen, debemos verificar si representan capturas duplicadas o incidentes legítimamente distintos.

In [5]:
folios_repetidos = [(folio, n) for folio, n in folios.items() if n > 1]
folios_repetidos.sort(key=lambda x: (-x[1], x[0]))
resumen_folios = pd.DataFrame([{
    'folios_distintos_utilizables': len(folios),
    'folios_utilizables_repetidos': len(folios_repetidos),
    'filas_en_folios_repetidos': sum(n for _, n in folios_repetidos),
    'maximo_filas_por_folio': max(folios.values()) if folios else 0,
    'duplicados_adicionales_17_columnas': metricas['duplicados_adicionales_17_columnas'],
    'huellas_de_grupos_identicos': len(huellas_repetidas),
    'filas_eliminadas': 0
}])
display(resumen_folios)
resumen_folios.to_csv(SALIDA / 'resumen_folios_duplicados_C5.csv',
                      index=False, encoding='utf-8-sig')
pd.DataFrame(folios_repetidos[:500], columns=['folio','numero_filas']).to_csv(
    SALIDA / 'folios_repetidos_top500_C5.csv', index=False, encoding='utf-8-sig')

,folios_distintos_utilizables,folios_utilizables_repetidos,filas_en_folios_repetidos,maximo_filas_por_folio,duplicados_adicionales_17_columnas,huellas_de_grupos_identicos,filas_eliminadas
0,1341409,10,20,2,0,0,0


## 7. Cobertura temporal y categorías
Guardamos la distribución por archivo, año y mes. El volumen de 2024 debe interpretarse considerando su cobertura real. Los tipos de incidente C5 no constituyen por sí solos un conteo confirmado de personas lesionadas o fallecidas.

In [6]:
def guardar_frecuencias(contador, columna, archivo):
    df = pd.DataFrame(list(contador.items()), columns=[columna,'filas'])
    df = df.sort_values(columna).reset_index(drop=True)
    df.to_csv(SALIDA / archivo, index=False, encoding='utf-8-sig')
    return df

por_archivo = guardar_frecuencias(archivos, 'archivo_origen', 'frecuencias_archivos_C5.csv')
por_anio = guardar_frecuencias(anios, 'anio', 'frecuencias_anios_C5_etapa2.csv')
por_mes = guardar_frecuencias(meses, 'mes', 'frecuencias_meses_C5.csv')
por_alcaldia = guardar_frecuencias(alcaldias, 'alcaldia', 'frecuencias_alcaldias_C5_etapa2.csv')
por_incidente = guardar_frecuencias(incidentes, 'incidente_c4', 'frecuencias_incidentes_C5_etapa2.csv')
display(por_archivo)
display(por_anio)
display(por_incidente.sort_values('filas', ascending=False).head(15))

,archivo_origen,filas
0,inViales_2016_2018.csv,246561
1,inViales_2019_2021.csv,590621
2,inViales_2022_2024.csv,504237


,anio,filas
0,2018,246577
1,2019,229494
2,2020,162671
3,2021,198513
4,2022,234914
5,2023,233325
6,2024,35925


,incidente_c4,filas
4,Choque sin lesionados,649483
2,Choque con lesionados,362075
1,Atropellado,154448
9,Motociclista,115150
16,Volcadura,28420
5,Ciclista,12496
11,Persona atrapada / desbarrancada,6792
15,Vehículo atrapadovarado,3869
3,Choque con prensados,2961
0,Accidente automovilístico,1682


## 8. Ejemplos para revisión manual
Exportamos hasta 100 ejemplos por alerta, con folio, procedencia y columnas originales. **No publicamos interpretaciones sobre las discrepancias antes de inspeccionar sus casos**.

In [7]:
for tipo, filas in muestras.items():
    df = pd.DataFrame(filas)
    df.to_csv(SALIDA / f'muestra_{tipo}_C5.csv',
              index=False, encoding='utf-8-sig')
muestras_resumen = pd.DataFrame([
    {'alerta': tipo, 'filas_en_muestra': len(filas)}
    for tipo, filas in muestras.items()
])
display(muestras_resumen)
muestras_resumen.to_csv(SALIDA / 'inventario_muestras_C5.csv',
                       index=False, encoding='utf-8-sig')

,alerta,filas_en_muestra
0,alcaldias_discrepantes,100
1,duplicados_adicionales,0
2,fecha_invalida,0
3,fecha_futura,0
4,hora_creacion_invalida,0
5,coordenadas_invalidas,0


## 9. Comprobaciones finales
Verificamos que se revisó exactamente el mismo número de filas que se exportó en la Etapa 1, que las frecuencias coinciden con el total y que **no hemos eliminado ni sobrescrito registros**. Si alguna comprobación falla, detenemos la etapa y revisamos los archivos.

In [8]:
esperado = 1341419  # Referencia de la ejecución de Etapa 1 comunicada por el equipo
comprobaciones = {
    'conteo_igual_a_etapa1_reportada': metricas['filas'] == esperado,
    'conteo_por_archivo': sum(archivos.values()) == metricas['filas'],
    'conteo_por_anio': sum(anios.values()) + metricas['fechas_invalidas'] == metricas['filas'],
    'conteo_por_alcaldia': sum(alcaldias.values()) == metricas['filas'],
    'conteo_por_incidente': sum(incidentes.values()) == metricas['filas'],
    'sin_filas_eliminadas': True,
    'archivo_entrada_intacto': ARCHIVO.is_file()
}
comprobaciones_df = pd.DataFrame([
    {'comprobacion': k, 'cumple': v} for k,v in comprobaciones.items()
])
display(comprobaciones_df)
comprobaciones_df.to_csv(SALIDA / 'comprobaciones_C5.csv',
                         index=False, encoding='utf-8-sig')
assert all(comprobaciones.values()), 'Revisar las comprobaciones antes de continuar.'
print('Comprobaciones superadas. Etapa 2 concluida sin eliminación de registros.')

,comprobacion,cumple
0,conteo_igual_a_etapa1_reportada,True
1,conteo_por_archivo,True
2,conteo_por_anio,True
3,conteo_por_alcaldia,True
4,conteo_por_incidente,True
5,sin_filas_eliminadas,True
6,archivo_entrada_intacto,True


Comprobaciones superadas. Etapa 2 concluida sin eliminación de registros.


## 10. Bitácora de decisiones y conclusión
**Lo que hicimos:** perfilamos campos, comprobamos fechas y horas, revisamos folios y duplicados exactos, identificamos discrepancias entre alcaldías y medimos calidad de coordenadas. Generamos archivos CSV de evidencia y conservamos la fuente delimitada intacta.

**Lo que todavía no decidimos:** eliminar duplicados; resolver conflictos entre alcaldías; asumir que categorías como «Lesionado» equivalen a cifras de víctimas; unir directamente incidentes C5 con hechos SSC.

**Siguiente etapa:** con estos resultados elaboraremos `03_etapa3_reglas_duplicados_C5.ipynb`, definiendo reglas respaldadas por las evidencias y justificando cualquier transformación. No retiraremos registros por folio repetido sin examinar la unidad de observación.

In [9]:
print("RESUMEN DE VALIDACIONES C5")
display(resumen)

print("\nFOLIOS Y DUPLICADOS C5")
display(resumen_folios)

RESUMEN DE VALIDACIONES C5


,indicador,filas,porcentaje_sobre_total
0,alcaldia_catalogo_norm_faltante,2345,0.17
1,alcaldia_cierre_norm_faltante,84,0.01
2,alcaldia_inicio_norm_faltante,102,0.01
3,alcaldia_normalizada_invalida,0,0.00
4,alcaldias_discrepantes,76204,5.68
5,coordenadas_fuera_rango_global,0,0.00
6,coordenadas_no_numericas_o_faltantes,0,0.00
7,duplicados_adicionales_17_columnas,0,0.00
8,fechas_antes_2018,0,0.00
9,fechas_futuras_alerta,0,0.00



FOLIOS Y DUPLICADOS C5


,folios_distintos_utilizables,folios_utilizables_repetidos,filas_en_folios_repetidos,maximo_filas_por_folio,duplicados_adicionales_17_columnas,huellas_de_grupos_identicos,filas_eliminadas
0,1341409,10,20,2,0,0,0


In [10]:
print("RESUMEN COMPLETO DE FOLIOS C5")

for columna in resumen_folios.columns:
    print(f"{columna}: {resumen_folios.iloc[0][columna]}")

RESUMEN COMPLETO DE FOLIOS C5
folios_distintos_utilizables: 1341409
folios_utilizables_repetidos: 10
filas_en_folios_repetidos: 20
maximo_filas_por_folio: 2
duplicados_adicionales_17_columnas: 0
huellas_de_grupos_identicos: 0
filas_eliminadas: 0
